In [1]:

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold

pd.set_option('display.max_columns', 100)

TRAIN_PATH = 'train.csv'
TEST_PATH = 'test.csv'
TARGET_COLUMN = 'SalePrice'
ID_COLUMN = 'Id'

RANDOM_STATE = 42  

In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f'train_df shape: {train_df.shape}')
print(f'test_df shape : {test_df.shape}')
train_df.head()

train_df shape: (1460, 81)
test_df shape : (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
def get_categorical_columns(df, exclude=None):

    exclude = exclude or []
    return [col for col in df.columns
            if pd.api.types.is_string_dtype(df[col]) and col not in exclude]

categorical_columns = get_categorical_columns(train_df, exclude=[ID_COLUMN, TARGET_COLUMN])
print(f'Found {len(categorical_columns)} categorical columns')

cardinality = train_df[categorical_columns].nunique().sort_values(ascending=False)
cardinality.to_frame('unique_values')

Found 27 categorical columns


,unique_values
Neighborhood,25
Exterior2nd,16
Exterior1st,15
Condition1,9
SaleType,9
RoofMatl,8
HouseStyle,8
Condition2,8
Functional,7
Foundation,6


In [4]:

NA_MEANS_NONE_COLUMNS = [
    'Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType',
    'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'
]

def fill_structural_missing(df, columns, fill_value='None'):
  
    df = df.copy()
    present_columns = [c for c in columns if c in df.columns]
    df[present_columns] = df[present_columns].fillna(fill_value)
    return df

def fill_remaining_categorical(df, columns):

    df = df.copy()
    for col in columns:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].mode()[0])
    return df

train_clean = fill_structural_missing(train_df, NA_MEANS_NONE_COLUMNS)
train_clean = fill_remaining_categorical(train_clean, categorical_columns)

test_clean = fill_structural_missing(test_df, NA_MEANS_NONE_COLUMNS)
test_clean = fill_remaining_categorical(test_clean, categorical_columns)

print('Remaining missing values in categorical columns (train):',
      train_clean[categorical_columns].isnull().sum().sum())

Remaining missing values in categorical columns (train): 0


In [5]:

demo_column = 'RoofStyle' 

label_encoder = LabelEncoder()
demo_before = train_clean[demo_column].head(8).reset_index(drop=True)
demo_after = pd.Series(label_encoder.fit_transform(train_clean[demo_column]), name=demo_column).head(8)

print('Categories found:', list(label_encoder.classes_))
pd.DataFrame({'before': demo_before, 'after_label_encoding': demo_after})

Categories found: ['Flat', 'Gable', 'Gambrel', 'Hip', 'Mansard', 'Shed']


,before,after_label_encoding
0,Gable,1
1,Gable,1
2,Gable,1
3,Gable,1
4,Gable,1
5,Gable,1
6,Gable,1
7,Gable,1


In [6]:
one_hot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
demo_one_hot = one_hot_encoder.fit_transform(train_clean[[demo_column]].head(8))
one_hot_columns = one_hot_encoder.get_feature_names_out([demo_column])

pd.DataFrame(demo_one_hot, columns=one_hot_columns)

,RoofStyle_Gable
0,1.0
1,1.0
2,1.0
3,1.0
4,1.0
5,1.0
6,1.0
7,1.0


In [7]:
QUALITY_SCALE = ['Po', 'Fa', 'TA', 'Gd', 'Ex']

QUALITY_SCALE_COLUMNS = [
    'ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual',
    'FireplaceQu', 'GarageQual', 'GarageCond']

QUALITY_SCALE_WITH_NONE = ['BsmtQual', 'BsmtCond', 'PoolQC']

def build_ordinal_encoder(base_scale, has_none=False):
    order = (['None'] + base_scale) if has_none else base_scale
    return OrdinalEncoder(categories=[order], handle_unknown='use_encoded_value', unknown_value=-1)

demo_quality_column = 'KitchenQual'
ordinal_encoder = build_ordinal_encoder(QUALITY_SCALE)
demo_ordinal = ordinal_encoder.fit_transform(train_clean[[demo_quality_column]].head(8))

pd.DataFrame({
    demo_quality_column: train_clean[demo_quality_column].head(8).reset_index(drop=True),
    'ordinal_code': demo_ordinal.flatten()
})

,KitchenQual,ordinal_code
0,Gd,3.0
1,TA,2.0
2,Gd,3.0
3,Gd,3.0
4,Gd,3.0
5,TA,2.0
6,Gd,3.0
7,TA,2.0


In [8]:
def fit_frequency_map(series):
    return series.value_counts(normalize=True).to_dict()

def apply_frequency_map(series, frequency_map, default=0.0):
    return series.map(frequency_map).fillna(default)

demo_freq_column = 'Neighborhood'
freq_map = fit_frequency_map(train_clean[demo_freq_column])
demo_freq_encoded = apply_frequency_map(train_clean[demo_freq_column], freq_map)

pd.DataFrame({
    demo_freq_column: train_clean[demo_freq_column].head(8).reset_index(drop=True),
    'frequency_encoded': demo_freq_encoded.head(8).reset_index(drop=True)
})

,Neighborhood,frequency_encoded
0,CollgCr,0.102740
1,Veenker,0.007534
2,CollgCr,0.102740
3,Crawfor,0.034932
4,NoRidge,0.028082
5,Mitchel,0.033562
6,Somerst,0.058904
7,NWAmes,0.050000


In [9]:

def fit_target_map(series, target):
    return target.groupby(series).mean().to_dict()

demo_target_column = 'Neighborhood'
target_map = fit_target_map(train_clean[demo_target_column], train_clean[TARGET_COLUMN])
demo_target_encoded = train_clean[demo_target_column].map(target_map)

pd.DataFrame({
    demo_target_column: train_clean[demo_target_column].head(8).reset_index(drop=True),
    'avg_SalePrice_for_category': demo_target_encoded.head(8).reset_index(drop=True).round(0)
})

,Neighborhood,avg_SalePrice_for_category
0,CollgCr,197966.0
1,Veenker,238773.0
2,CollgCr,197966.0
3,Crawfor,210625.0
4,NoRidge,335295.0
5,Mitchel,156270.0
6,Somerst,225380.0
7,NWAmes,189050.0


In [10]:
def split_columns_by_cardinality(columns, cardinality, low_cardinality_max):
  
    low = [c for c in columns if cardinality[c] <= low_cardinality_max]
    high = [c for c in columns if cardinality[c] > low_cardinality_max]
    return low, high

LOW_CARDINALITY_MAX = 10  

ordinal_quality_columns = QUALITY_SCALE_COLUMNS + QUALITY_SCALE_WITH_NONE
nominal_columns = [c for c in categorical_columns if c not in ordinal_quality_columns]
low_card_nominal, high_card_nominal = split_columns_by_cardinality(
    nominal_columns, cardinality, LOW_CARDINALITY_MAX
)

print(f'Ordinal (quality-scale) columns : {len(ordinal_quality_columns)}')
print(f'Low-cardinality nominal columns : {len(low_card_nominal)}')
print(f'High-cardinality nominal columns: {len(high_card_nominal)} -> {high_card_nominal}')

Ordinal (quality-scale) columns : 10
Low-cardinality nominal columns : 20
High-cardinality nominal columns: 3 -> ['Neighborhood', 'Exterior1st', 'Exterior2nd']


In [11]:

def apply_pathway_A(df, freq_maps=None, fit=True):
    df = df.copy()
    freq_maps = freq_maps or {}

    for col in QUALITY_SCALE_COLUMNS:
        encoder = build_ordinal_encoder(QUALITY_SCALE, has_none=False)
        df[col] = encoder.fit_transform(df[[col]])
    for col in QUALITY_SCALE_WITH_NONE:
        encoder = build_ordinal_encoder(QUALITY_SCALE, has_none=True)
        df[col] = encoder.fit_transform(df[[col]])

    df = pd.get_dummies(df, columns=low_card_nominal, drop_first=False)

    for col in high_card_nominal:
        if fit:
            freq_maps[col] = fit_frequency_map(df[col])
        df[col] = apply_frequency_map(df[col], freq_maps[col])

    return df, freq_maps

def apply_pathway_B(df):
    df = df.copy()
    for col in categorical_columns:
        df[col] = LabelEncoder().fit_transform(df[col])
    return df

def apply_pathway_C(df):
    return pd.get_dummies(df, columns=categorical_columns, drop_first=False)

In [12]:
pathway_A_df, freq_maps_A = apply_pathway_A(train_clean)
pathway_B_df = apply_pathway_B(train_clean)
pathway_C_df = apply_pathway_C(train_clean)

pd.DataFrame({
    'pathway': ['A: mixed (ordinal + one-hot + frequency)', 'B: label-encode everything', 'C: one-hot everything'],
    'resulting_columns': [pathway_A_df.shape[1], pathway_B_df.shape[1], pathway_C_df.shape[1]]
})

,pathway,resulting_columns
0,A: mixed (ordinal + one-hot + frequency),169
1,B: label-encode everything,81
2,C: one-hot everything,236


In [13]:
def prepare_features(df, target_column, id_column):
    # 1. Separate features and target
    X = df.drop(columns=[target_column, id_column], errors='ignore')
    y = df[target_column]
    
    # 2. Impute missing numbers
    numeric_columns = X.select_dtypes(include=[np.number]).columns
    if not numeric_columns.empty:
        X[numeric_columns] = SimpleImputer(strategy='median').fit_transform(X[numeric_columns])
    
    # 3. 🚨 FIX: Only keep numeric columns (drop unencoded text columns)
    X = X[numeric_columns]
    
    return X, y

def evaluate_pathway(df, target_column, id_column, model, cv_folds=5, random_state=RANDOM_STATE):
    X, y = prepare_features(df, target_column, id_column)
    y_log = np.log1p(y)
    cv = KFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    scores = cross_val_score(model, X, y_log, cv=cv, scoring='neg_root_mean_squared_error')
    return -scores.mean()

pathways = {
    'A: mixed': pathway_A_df,
    'B: label-encode all': pathway_B_df,
    'C: one-hot all': pathway_C_df,
}

models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
}

results = []
for pathway_name, encoded_df in pathways.items():
    for model_name, model in models.items():
        rmse = evaluate_pathway(encoded_df, TARGET_COLUMN, ID_COLUMN, model)
        results.append({
            'pathway': pathway_name, 
            'model': model_name, 
            'cv_rmse_log_saleprice': round(rmse, 4)
        })

results_df = pd.DataFrame(results).sort_values('cv_rmse_log_saleprice')
results_df

,pathway,model,cv_rmse_log_saleprice
3,B: label-encode all,RandomForest,0.1434
1,A: mixed,RandomForest,0.1453
5,C: one-hot all,RandomForest,0.1478
2,B: label-encode all,LinearRegression,0.1574
0,A: mixed,LinearRegression,0.1592
4,C: one-hot all,LinearRegression,0.1599


In [14]:
results_df.to_csv('model_results.csv', index=True)
